In [1]:
import pandas as pd
import torch
import numpy as np


states = torch.load("states.pt")
target = pd.read_csv("target.csv")

In [2]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [3]:
row = states.shape[0]

print(states.shape)
print(target)

torch.Size([707542, 30, 8, 8])
        value  policy
0          -1     307
1           1    4488
2          -1     657
3           1    4395
4          -1     195
...       ...     ...
707537      1    2538
707538     -1    4488
707539      1    2847
707540     -1    2604
707541      1    3480

[707542 rows x 2 columns]


In [4]:
import sys
sys.path.append('..')

In [5]:
policy = torch.tensor(target.policy.values).float()
value = torch.tensor(target.value.values).float()


In [6]:
# Train test split
TRAIN_SIZE = int(0.9 * len(states)) 

train_states, test_states = states[:TRAIN_SIZE], states[TRAIN_SIZE:]
train_policy, test_policy = policy[:TRAIN_SIZE], policy[TRAIN_SIZE:]
train_value,  test_value  = value[:TRAIN_SIZE],  value[TRAIN_SIZE:]


In [7]:

from core import factory

network = factory.build_network("chess")

In [8]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = Adam(network.parameters(), lr=3e-4, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [9]:
from core.network import PolicyValueNetwork
from torch import optim
import time
from core.network import PolicyValueNetwork
from torch import optim
import time
import torch
import numpy as np
from torch.optim import lr_scheduler


def evaluate(network, test_states, test_policy, test_value,
             policy_loss_fn, value_loss_fn, batch_size=256):
    network.eval()
    total_policy_loss, total_value_loss, n_batches = 0.0, 0.0, 0

    with torch.no_grad():
        for i in range(0, len(test_states), batch_size):
            batch_states = test_states[i:i+batch_size]
            batch_policy = test_policy[i:i+batch_size]
            batch_value  = test_value[i:i+batch_size]

            policy_head, value_head = network(batch_states)
            total_policy_loss += policy_loss_fn(policy_head, batch_policy).item()
            total_value_loss  += value_loss_fn(value_head, batch_value).item()
            n_batches += 1

    network.train()
    return total_policy_loss / n_batches, total_value_loss / n_batches


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          train_states: torch.Tensor,
          train_policy: torch.Tensor,
          train_value: torch.Tensor,
          policy_loss_fn,
          value_loss_fn,
          test_states: torch.Tensor | None = None,
          test_policy: torch.Tensor | None = None,
          test_value: torch.Tensor | None = None,
          batch_size: int = 256,
          num_iter: int | None = None,
          duration_hour: float | None = None,
          eval_every: int = 500,
          seed: int = 42):

    if num_iter is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_iter or duration_hour")

    start = time.time()
    rng = np.random.default_rng(seed=seed)
    step = 0

    warmup_steps = 200

    warmup = lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
    cosine = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_iter - warmup_steps if num_iter else 9999)
    scheduler = lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

    train_policy = train_policy.to(device=DEVICE)
    train_value = train_value.unsqueeze(-1).to(device=DEVICE)

    has_test = test_states is not None and test_policy is not None and test_value is not None
    if has_test:
        test_states = test_states.to(device=DEVICE)
        test_policy = test_policy.to(device=DEVICE)
        test_value  = test_value.unsqueeze(-1).to(device=DEVICE)

    while True:
        if num_iter is not None and step >= num_iter:
            break
        if duration_hour is not None and time.time() - start >= duration_hour * 3600:
            break

        batch_idx = rng.choice(len(train_states), batch_size, replace=False)
        batch_train_states = train_states[batch_idx]
        batch_train_policy = train_policy[batch_idx]
        batch_train_value  = train_value[batch_idx]

        optimizer.zero_grad()
        policy_head, value_head = network(batch_train_states)

        policy_loss = policy_loss_fn(policy_head, batch_train_policy)
        value_loss  = value_loss_fn(value_head, batch_train_value)
        loss = policy_loss + value_loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        if step % 10 == 0:
            elapsed = time.time() - start
            print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={value_loss.item():.4f} | {elapsed:.0f}s")

        if has_test and step % eval_every == 0 and step > 0:
            val_policy_loss, val_value_loss = evaluate(
                network, test_states, test_policy, test_value, policy_loss_fn, value_loss_fn
            )
            print(f"    [eval @ {step}] val_policy={val_policy_loss:.4f} | val_value={val_value_loss:.4f}")

        step += 1

In [ ]:
n_train = int(0.8 * len(states))
train_states, test_states = states[:n_train], states[n_train:]
train_policy, test_policy = policy[:n_train], policy[n_train:]
train_value,  test_value  = value[:n_train],  value[n_train:]

train(
    duration_hour=1,
    network=network,
    optimizer=optimizer,
    train_states=train_states,
    train_policy=train_policy,
    train_value=train_value,
    test_states=test_states,
    test_policy=test_policy,
    test_value=test_value,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=32,
    seed=42
)

[0] loss=11.4258 | policy=10.1102 | value=1.3156 | 1s
[10] loss=10.7979 | policy=9.8130 | value=0.9849 | 5s
[20] loss=11.2557 | policy=9.7309 | value=1.5248 | 9s
[30] loss=10.7724 | policy=9.5941 | value=1.1783 | 14s
[40] loss=10.2392 | policy=9.3330 | value=0.9061 | 18s
[50] loss=9.4671 | policy=8.7768 | value=0.6903 | 23s
[60] loss=9.5202 | policy=8.7364 | value=0.7838 | 27s
[70] loss=10.2148 | policy=9.1679 | value=1.0468 | 31s
[80] loss=8.4599 | policy=7.8287 | value=0.6312 | 36s
[90] loss=9.3256 | policy=8.5800 | value=0.7456 | 41s
[100] loss=8.8667 | policy=8.0262 | value=0.8405 | 45s
[110] loss=8.3853 | policy=7.7672 | value=0.6181 | 50s
[120] loss=8.7266 | policy=8.1247 | value=0.6020 | 55s
[130] loss=8.2673 | policy=7.5671 | value=0.7002 | 60s
[140] loss=7.8098 | policy=7.1815 | value=0.6282 | 64s
[150] loss=7.3155 | policy=7.0434 | value=0.2721 | 68s
[160] loss=7.2925 | policy=6.8698 | value=0.4227 | 72s
[170] loss=7.8527 | policy=7.3111 | value=0.5416 | 77s
[180] loss=7.5573

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:243: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[200] loss=7.6628 | policy=7.0334 | value=0.6294 | 92s
[210] loss=7.6260 | policy=6.7999 | value=0.8261 | 96s
[220] loss=7.5792 | policy=7.0552 | value=0.5240 | 101s
[230] loss=7.4246 | policy=6.6929 | value=0.7317 | 106s
[240] loss=7.9772 | policy=7.4905 | value=0.4866 | 111s
[250] loss=7.7616 | policy=7.1017 | value=0.6600 | 116s
[260] loss=8.0000 | policy=7.2728 | value=0.7272 | 120s
[270] loss=7.4764 | policy=6.9527 | value=0.5237 | 124s
[280] loss=7.5611 | policy=6.8124 | value=0.7487 | 128s
[290] loss=6.6403 | policy=6.2813 | value=0.3591 | 132s
[300] loss=6.8378 | policy=6.3992 | value=0.4385 | 137s
[310] loss=7.2412 | policy=6.4788 | value=0.7623 | 141s
[320] loss=7.0511 | policy=6.7524 | value=0.2986 | 146s
[330] loss=6.6902 | policy=6.3015 | value=0.3887 | 151s
[340] loss=7.5556 | policy=7.1463 | value=0.4093 | 156s
[350] loss=6.9320 | policy=6.1936 | value=0.7384 | 160s
[360] loss=7.0857 | policy=6.4949 | value=0.5908 | 165s
[370] loss=7.4680 | policy=6.8798 | value=0.5882 |